# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SalehAl-Nassar/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.**  
Unit: one content page. Task: score/rank pages by refresh priority.  
Proxy label: a page whose GSC position worsened from H1 to H2 of the same month.

## 1. Unit of analysis + time window

**Unit:** one content page (`content_hash_id`), aggregated to the page-month level.  
**Feature window (H1):** March 1-15, 2026 (days 1-15). All feature columns are computed from this window.  
**Label window (H2):** March 16-31, 2026 (days 16-31). The proxy label is computed from this window.  
**Why a temporal split?** Features must be knowable *before* the label window opens. A same-window proxy (like the starter CSV's `trend_direction == 'down'`) leaks the future into the present. Splitting each month into H1->H2 ensures temporal order without requiring historical snapshots.  

Mid-panel month: **2026-03** (March), 9,841,378 daily rows across ~331K distinct content items and 55 clients.  
Test/sealed month: **2026-06** (June, `_sample` table) — used only for final evaluation, never for label development.

In [ ]:
import os, duckdb
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

# -- token -----------------------------------------------------------
token = None
from pathlib import Path
for p in [Path('../../.env'), Path('.env'), Path.home() / '.env']:
    if p.exists():
        with open(p) as f:
            for line in f:
                l = line.strip()
                if l.startswith('hf_token='):
                    token = l.split('=',1)[1].strip()
                    break
        break

# -- DuckDB + HF secret ----------------------------------------------
con = duckdb.connect()
con.execute('INSTALL httpfs; LOAD httpfs;')
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{}')".format(token))

base = 'hf://datasets/FlyRank/internship-warehouse'
url_mar = '{}/fact_content_daily_performance/month=2026-03/data_0.parquet'.format(base)

# -- 1a. verify March partition --------------------------------------
r = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id) AS distinct_pages,
        COUNT(DISTINCT client_hash_id) AS distinct_clients,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet('{url}')
""".format(url=url_mar)).to_df()
print('March 2026 partition - overview')
print(r.to_string(index=False))
print()

# -- 1b. verify H1 / H2 split ---------------------------------------
r2 = con.sql("""
    SELECT
        CASE WHEN EXTRACT(DAY FROM report_date) BETWEEN 1 AND 15 THEN 'H1 (1-15)'
             WHEN EXTRACT(DAY FROM report_date) BETWEEN 16 AND 31 THEN 'H2 (16-31)'
             ELSE 'other' END AS half,
        COUNT(*) AS rows,
        COUNT(DISTINCT content_hash_id) AS pages,
        MIN(report_date) AS min_dt,
        MAX(report_date) AS max_dt
    FROM read_parquet('{url}')
    GROUP BY half
    ORDER BY half
""".format(url=url_mar)).to_df()
print('H1 / H2 split')
print(r2.to_string(index=False))
print()

# -- 1c. cache page-month aggregate ---------------------------------
print('Caching page-month aggregate to work/outputs/mar_page_month.parquet ...')
con.sql("""
    CREATE OR REPLACE TABLE h1 AS
        SELECT content_hash_id, client_hash_id,
               SUM(gsc_impressions) AS impressions_h1,
               SUM(gsc_clicks) AS clicks_h1,
               AVG(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position ELSE NULL END) AS avg_position_h1,
               SUM(ga4_sessions) AS sessions_h1,
               SUM(ga4_engaged_sessions) AS engaged_sessions_h1,
               SUM(sessions_organic) AS sessions_organic_h1
        FROM read_parquet('{url}')
        WHERE EXTRACT(DAY FROM report_date) BETWEEN 1 AND 15
        GROUP BY content_hash_id, client_hash_id;
    CREATE OR REPLACE TABLE h2 AS
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions_h2,
               SUM(gsc_clicks) AS clicks_h2,
               AVG(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position ELSE NULL END) AS avg_position_h2,
               SUM(ga4_sessions) AS sessions_h2,
               SUM(ga4_engaged_sessions) AS engaged_sessions_h2,
               SUM(sessions_organic) AS sessions_organic_h2
        FROM read_parquet('{url}')
        WHERE EXTRACT(DAY FROM report_date) BETWEEN 16 AND 31
        GROUP BY content_hash_id;
    CREATE OR REPLACE TABLE page_month AS
        SELECT h1.content_hash_id, h1.client_hash_id,
               h1.impressions_h1, h1.clicks_h1, h1.avg_position_h1,
               h1.sessions_h1, h1.engaged_sessions_h1, h1.sessions_organic_h1,
               h2.impressions_h2, h2.clicks_h2, h2.avg_position_h2,
               h2.sessions_h2, h2.engaged_sessions_h2, h2.sessions_organic_h2,
               dc.word_count, dc.search_volume, dc.competition_level,
               dc.main_intent, dc.content_type, dc.content_created_date,
               dc.last_optimized_date,
               DATEDIFF('day', COALESCE(dc.content_created_date, '2026-03-01'::DATE), '2026-03-01'::DATE) AS content_age_days,
               GREATEST(0, DATEDIFF('day', COALESCE(dc.last_optimized_date, dc.content_created_date, '2026-03-01'::DATE), '2026-03-01'::DATE)) AS days_since_update
        FROM h1
        INNER JOIN h2 ON h1.content_hash_id = h2.content_hash_id
        LEFT JOIN read_parquet('{base}/dim_content.parquet') dc
               ON h1.content_hash_id = dc.content_hash_id
""".format(url=url_mar, base=base))
con.sql("COPY page_month TO 'work/outputs/mar_page_month.parquet' (FORMAT PARQUET)")

r3 = con.sql('SELECT COUNT(*) AS n FROM page_month').to_df()
print('Cached: {} page-month rows'.format(r3.n[0]))
con.sql('SELECT * FROM page_month LIMIT 3').to_df()

## 2. Fields: feature / label / context / excluded

### Features (knowable before H2 opens)

| Field | Source | Description |
|---|---|---|
| `impressions_h1` | fact (days 1-15) | Total GSC impressions in the feature window |
| `clicks_h1` | fact (days 1-15) | Total GSC clicks |
| `avg_position_h1` | fact (days 1-15) | Average GSC position (lower = better). ~52% NULL for pages with zero impressions in H1 |
| `sessions_h1` | fact (days 1-15) | GA4 sessions. ~31% NULL where `ga4_data_available = FALSE` |
| `engaged_sessions_h1` | fact (days 1-15) | GA4 engaged sessions |
| `sessions_organic_h1` | fact (days 1-15) | Organic search sessions |
| `word_count` | dim_content | Content length in words. ~34% NULL (pages without dim_content match) |
| `search_volume` | dim_content | Keyword-level search demand. ~18% NULL |
| `competition_level` | dim_content | Keyword competition (LOW / MEDIUM / HIGH). ~18% NULL |
| `main_intent` | dim_content | Keyword intent (informational, commercial, transactional, navigational). ~18% NULL |
| `content_type` | dim_content | Type of content page. 0% NULL |
| `content_age_days` | derived | Days since `content_created_date` to March 1, 2026. 0% NULL |
| `days_since_update` | derived | Days since `last_optimized_date` (or created date) to March 1. Capped at 0. |

### Label / Proxy (from H2, days 16-31)

| Field | Description |
|---|---|
| `impressions_h2`, `clicks_h2`, `avg_position_h2` | H2 metrics used to compute the proxy |
| `proxy_decline` (derived) | **1** if avg_position worsened >=10% from H1 to H2, **0** otherwise. Favors recall over precision. Observed rate: ~44% of pages with non-NULL position in both halves. |

### Context (join / group / split only — never features)

`content_hash_id`, `client_hash_id` — pseudonyms used for grouping, joining, and per-client train/test splits.

### Excluded

| Field | Why excluded |
|---|---|
| `report_date` | Partition column; each row is already page-month |
| `month` | Redundant partition label |
| `keyword_hash_id`, `url_hash_id` | Raw identifiers, not signals |
| `provider_used`, `model_used` | Product implementation detail; not a content signal |
| `is_published`, `is_deleted` | Product status flags, not performance signals |
| `ga4_data_available`, `gsc_data_available` | Gating flags, not features — but they *are* checked to understand missingness patterns (see Section 4) |

In [ ]:
# -- 2a. reload cached page-month ---------------------------------
con2 = duckdb.connect()
con2.execute('INSTALL httpfs; LOAD httpfs;')
con2.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{}')".format(token))
con2.execute("CREATE OR REPLACE VIEW pm AS SELECT * FROM read_parquet('work/outputs/mar_page_month.parquet')")

# -- 2b. show columns actually present ------------------------------
print('Columns in page-month cache:')
print([c[0] for c in con2.sql('DESCRIBE pm').fetchall()])
print()

# -- 2c. missingness overview ---------------------------------------
r_miss = con2.sql("""
    SELECT
        COUNT(*) AS total_rows,
        AVG(CASE WHEN impressions_h1 IS NULL THEN 1.0 ELSE 0 END) AS miss_impressions_h1,
        AVG(CASE WHEN clicks_h1 IS NULL THEN 1.0 ELSE 0 END) AS miss_clicks_h1,
        AVG(CASE WHEN avg_position_h1 IS NULL THEN 1.0 ELSE 0 END) AS miss_avg_pos_h1,
        AVG(CASE WHEN sessions_h1 IS NULL THEN 1.0 ELSE 0 END) AS miss_sessions_h1,
        AVG(CASE WHEN word_count IS NULL THEN 1.0 ELSE 0 END) AS miss_word_count,
        AVG(CASE WHEN search_volume IS NULL THEN 1.0 ELSE 0 END) AS miss_search_volume,
        AVG(CASE WHEN competition_level IS NULL THEN 1.0 ELSE 0 END) AS miss_competition,
        AVG(CASE WHEN main_intent IS NULL THEN 1.0 ELSE 0 END) AS miss_intent,
        AVG(CASE WHEN content_type IS NULL THEN 1.0 ELSE 0 END) AS miss_content_type,
        AVG(CASE WHEN content_age_days IS NULL THEN 1.0 ELSE 0 END) AS miss_age,
        AVG(CASE WHEN days_since_update IS NULL THEN 1.0 ELSE 0 END) AS miss_days_since_update
    FROM pm
""").to_df()
print('Missingness rates:')
for col in r_miss.columns:
    v = r_miss[col].values[0]
    if col != 'total_rows':
        print('  {}: {:.4f} ({:.1f}%)'.format(col, v, v * 100))
print()

# -- 2d. sample rows ------------------------------------------------
con2.sql('SELECT * FROM pm LIMIT 3').to_df()

## 3. Verify it with queries (grain, counts, missing values, windows)

Every claim in sections 1 and 2 is checked below. No sentence without a supporting query.

In [ ]:
# -- 3a. Grain check -----------------------------------------------
print('=== Grain: rows with duplicate content_hash_id ===')
bad = con2.sql("""
    SELECT content_hash_id, COUNT(*) AS c
    FROM pm
    GROUP BY content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").to_df()
if len(bad) == 0:
    print('  PASS -- no duplicates found. Grain holds: 1 row = 1 page.')
else:
    print('  FAIL -- {} duplicates exist'.format(len(bad)))
    print(bad)
print()

# -- 3b. Pages per client -------------------------------------------
print('=== Pages per client (top 5) ===')
print(con2.sql("""
    SELECT client_hash_id, COUNT(*) AS pages
    FROM pm
    GROUP BY client_hash_id
    ORDER BY pages DESC
    LIMIT 5
""").to_df().to_string(index=False))
print()

# -- 3c. H1 / H2 stats ----------------------------------------------
print('=== H1 stats (days 1-15 of March 2026) ===')
print(con2.sql("""
    SELECT
        COUNT(*) AS pages,
        MIN(impressions_h1) AS min_imp,
        AVG(impressions_h1) AS avg_imp,
        MAX(impressions_h1) AS max_imp,
        AVG(clicks_h1) AS avg_clicks,
        AVG(avg_position_h1) AS avg_pos
    FROM pm
""").to_df().to_string(index=False))
print()

print('=== H2 stats (days 16-31 of March 2026) ===')
print(con2.sql("""
    SELECT
        COUNT(*) AS pages,
        MIN(impressions_h2) AS min_imp,
        AVG(impressions_h2) AS avg_imp,
        MAX(impressions_h2) AS max_imp,
        AVG(clicks_h2) AS avg_clicks,
        AVG(avg_position_h2) AS avg_pos
    FROM pm
""").to_df().to_string(index=False))
print()

# -- 3d. dim_content missingness by client --------------------------
print('=== Rows with NULL word_count (missing dim_content join) ===')
r_miss_dc = con2.sql("""
    SELECT
        COUNT(*) AS total_nodc,
        COUNT(DISTINCT client_hash_id) AS clients_nodc
    FROM pm
    WHERE word_count IS NULL
""").to_df()
total_rows = r_miss.total_rows[0]
print('  Pages without dim_content: {} ({:.1f}%)'.format(r_miss_dc.total_nodc[0], r_miss_dc.total_nodc[0] / total_rows * 100))
print('  Affected clients: {}'.format(r_miss_dc.clients_nodc[0]))
print()

# -- 3e. Proxy label distribution -----------------------------------
print('=== Proxy label: avg_position worsened >=10% from H1 to H2 ===')
r_label = con2.sql("""
    SELECT
        CASE WHEN avg_position_h2 > avg_position_h1 * 1.10 THEN 1 ELSE 0 END AS proxy_decline,
        COUNT(*) AS pages
    FROM pm
    WHERE avg_position_h1 IS NOT NULL AND avg_position_h2 IS NOT NULL
    GROUP BY proxy_decline
    ORDER BY proxy_decline
""").to_df()
print(r_label.to_string(index=False))
if len(r_label) >= 2:
    total = r_label.pages.sum()
    declined = r_label[r_label.proxy_decline == 1].pages.values[0]
    print('  Proxy decline rate: {}/{} = {:.1f}%'.format(declined, total, declined / total * 100))
print()

# -- 3f. Confirm no gsc_ctr column ----------------------------------
print('=== Confirming: no gsc_ctr column exists ===')
base_url = 'hf://datasets/FlyRank/internship-warehouse'
r_has_ctr = con2.sql("""
    SELECT column_name
    FROM (SELECT * FROM read_parquet('{}/fact_content_daily_performance_sample.parquet') LIMIT 0)
    WHERE column_name = 'gsc_ctr'
""".format(base_url)).to_df()
if len(r_has_ctr) == 0:
    print('  CONFIRMED: gsc_ctr is not a warehouse column. Must be computed as clicks / impressions.')
else:
    print('  UNEXPECTED: gsc_ctr exists!')

## 4. Data limits

*What this data can never tell you — and why it matters for the contract.*

### 4a. GA4 history starts later than GSC for most clients
Only ~4% of March 2026 rows have `ga4_data_available = TRUE`. The rest are either FALSE (65%) or NULL/NA (31%). This means GA4-based features (`sessions_h1`, `engaged_sessions_h1`) will be NULL for ~96% of the rows in this architecture. A feature set that depends on GA4 will be limited to a small subset of pages. A practical mitigation: use `has_ga4` flags and GSC-only features as the primary set.

### 4b. Pages dropped from H2 get no label
The page-month aggregate inner-joins H1 and H2. Only 1 page out of 331K is lost, so survivorship bias is negligible for this mid-panel month.

### 4c. Temporal split reduces usable rows for the proxy label
The proxy label requires non-NULL `avg_position` in both H1 and H2. With ~52.5% of pages having NULL avg_position_h1 (zero impressions), only ~141K of 320K pages (44%) receive a label. The label rate among those is ~44% declined.

### 4d. Proxy label is directional, not ground truth
`proxy_decline = 1` means the page's avg rank number got >=10% higher (worse). This is a heuristic, not a Google update or user satisfaction signal. A true content-refresh label would need editorial judgments — out of scope for this project.

### 4e. Single mid-panel month for development
We develop on March 2026 only. The final month (June 2026) is sealed. Patterns that hold in March may not generalise — especially seasonal content or algorithm changes.

### 4f. GSC ctr is computed, not stored
There is no `gsc_ctr` column. CTR must be derived as `clicks / NULLIF(impressions, 0)`. Verified in 3f.

### 4g. days_since_update can be negative
`last_optimized_date` can fall after March 1 (future-dated). Capped at 0 with `GREATEST(0, ...)`.

### 4h. Registration-day trap (accrual from creation date)
Pages created after March 1 have zero history before their `content_created_date`. Their early H1 days show zero impressions — not bad performance, but absence. This inflates the pool of 'zero-impression' pages. Verified: ~5.3% of cached pages have `content_created_date` after March 1.

### 4i. GSC CTR rate convention
In the starter CSV, `ctr = 0.76` means 0.76% (×100). Our warehouse-derived CTR (`clicks / impressions`) is a raw ratio: 0.0076. Neither is wrong, but downstream code must pick one convention and apply it consistently.

In [ ]:
# -- 4a. GA4 data availability -------------------------------------
print('=== GA4 data availability (raw March 2026) ===')
r_ga4 = con.sql("""
    SELECT
        ga4_data_available,
        COUNT(*) AS rows,
        COUNT(DISTINCT content_hash_id) AS pages,
        COUNT(DISTINCT client_hash_id) AS clients
    FROM read_parquet('{url}')
    GROUP BY ga4_data_available
    ORDER BY ga4_data_available
""".format(url=url_mar)).to_df()
print(r_ga4.to_string(index=False))
print()

# -- 4b. H1-only pages (no H2 label) ---------------------------------
print('=== Pages with data in H1 but not H2 (no label possible) ===')
r_h1_only = con.sql("""
    SELECT COUNT(*) AS h1_only_pages
    FROM (
        SELECT content_hash_id
        FROM read_parquet('{url}')
        WHERE EXTRACT(DAY FROM report_date) BETWEEN 1 AND 15
        GROUP BY content_hash_id
    ) h1
    WHERE content_hash_id NOT IN (
        SELECT content_hash_id
        FROM read_parquet('{url}')
        WHERE EXTRACT(DAY FROM report_date) BETWEEN 16 AND 31
        GROUP BY content_hash_id
    )
""".format(url=url_mar)).to_df()
r_mar_pages = con.sql("""
    SELECT COUNT(DISTINCT content_hash_id) AS n_mar
    FROM read_parquet('{url}')
""".format(url=url_mar)).to_df()
print('  Pages in March with any data: {}'.format(r_mar_pages.n_mar[0]))
print('  Pages with H1 data only (no H2): {} ({:.1f}%)'.format(
    r_h1_only.h1_only_pages[0], r_h1_only.h1_only_pages[0] / r_mar_pages.n_mar[0] * 100))
print()

# -- 4c. Pages with very few H1 days ---------------------------------
print('=== Pages with <3 days of data in H1 ===')
r_noisy = con.sql("""
    SELECT COUNT(*) AS pages_noisy
    FROM (
        SELECT content_hash_id, COUNT(*) AS days_in_h1
        FROM read_parquet('{url}')
        WHERE EXTRACT(DAY FROM report_date) BETWEEN 1 AND 15
        GROUP BY content_hash_id
        HAVING COUNT(*) < 3
    )
""".format(url=url_mar)).to_df()
print('  Pages with <3 H1 days: {} ({:.1f}%)'.format(
    r_noisy.pages_noisy[0], r_noisy.pages_noisy[0] / r_mar_pages.n_mar[0] * 100))

# -- 4d. Registration-day trap ----------------------------------------
print('=== Pages created after March 1 (zero-history days in H1) ===')
r_created_after = con.sql("""
    SELECT COUNT(*) AS pages_late_start
    FROM read_parquet('work/outputs/mar_page_month.parquet')
    WHERE content_created_date > '2026-03-01'::DATE
""").to_df()
r_total_pm = con.sql("""
    SELECT COUNT(*) AS n FROM read_parquet('work/outputs/mar_page_month.parquet')
""").to_df()
print('  Pages with content_created_date after March 1: {} ({:.1f}%)'.format(
    r_created_after.pages_late_start[0], r_created_after.pages_late_start[0] / r_total_pm.n[0] * 100))
print()

# -- 4e. Fact vs dim_content client gap -------------------------------
print('=== Clients in fact table vs dim_content ===')
r_clients_fact = con.sql("""
    SELECT COUNT(DISTINCT client_hash_id) AS n FROM read_parquet('{url}')
""".format(url=url_mar)).to_df()
r_clients_dc = con.sql("""
    SELECT COUNT(DISTINCT client_hash_id) AS n FROM read_parquet('{base}/dim_content.parquet')
""".format(base=base)).to_df()
print('  Clients in March fact table: {}'.format(r_clients_fact.n[0]))
print('  Clients in dim_content: {}'.format(r_clients_dc.n[0]))
print('  Gap: {} clients have dim_content entries but no March fact data.'.format(
    r_clients_dc.n[0] - r_clients_fact.n[0]))

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.